**Note: Run servers on local instead of Colab**

Here are some comments for a tutorial on Model serialization with Pickle/Joblib:

1. **Introduction to Model Serialization**:
   - Model serialization is the process of converting a machine learning model into a format that can be easily saved and loaded. This is crucial for deploying models in production environments or sharing them with others
.

2. **Why Use Pickle or Joblib?**:
   - Pickle and Joblib are two popular libraries in Python for serializing machine learning models. Pickle is a built-in Python module that can serialize and deserialize any Python object, including custom classes and objects
.
   - Joblib is more efficient than Pickle when working with large numpy arrays, making it a better choice for serializing large machine learning models
.

3. **Security Considerations**:
   - Both Pickle and Joblib use the pickle protocol under the hood, which can execute arbitrary code during unpickling. Therefore, it is important to only load models from trusted sources to avoid security risks
.

4. **Saving a Model with Pickle**:
   - To save a model using Pickle, you can use the `pickle.dump` function. This function serializes the model and writes it to a file. For example:
     ```python
     import pickle
     with open('model.pkl', 'wb') as file:
         pickle.dump(model, file)
     ```
   - This code opens a file in write-binary mode and uses Pickle to serialize the model into this file
.

5. **Loading a Model with Pickle**:
   - To load a model using Pickle, you can use the `pickle.load` function. This function reads the serialized model from a file and deserializes it. For example:
     ```python
     with open('model.pkl', 'rb') as file:
         model = pickle.load(file)
     ```
   - This code opens the file in read-binary mode and uses Pickle to deserialize the model from this file
.

6. **Saving a Model with Joblib**:
   - To save a model using Joblib, you can use the `joblib.dump` function. This function serializes the model and writes it to a file. For example:
     ```python
     from joblib import dump, load
     dump(model, 'model.joblib')
     ```
   - This code uses Joblib to serialize the model into a file named 'model.joblib'
.

7. **Loading a Model with Joblib**:
   - To load a model using Joblib, you can use the `joblib.load` function. This function reads the serialized model from a file and deserializes it. For example:
     ```python
     model = load('model.joblib')
     ```
   - This code uses Joblib to deserialize the model from the file named 'model.joblib'
.

8. **Performance Comparison**:
   - Joblib is generally faster than Pickle when dealing with large numpy arrays due to its optimizations for such data structures. However, Pickle may be faster for smaller models or models that do not contain large numpy arrays
.

9. **Use Cases**:
   - Use Pickle when you need to serialize small models or models that do not contain large numpy arrays. Use Joblib when you need to serialize large models or models that contain large numpy arrays
.

10. **Conclusion**:
    - Both Pickle and Joblib are powerful tools for model serialization. The choice between them depends on the size of your model and the data structures it contains. Always ensure that you load models from trusted sources to avoid security risks
.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])
pipeline.fit(X_train, y_train)


,steps,"[('scaler', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2


In [3]:
import pickle
version = '1.0.0'
filename_pkl = f'model_v{version}.pkl'
with open(filename_pkl, 'wb') as f:
    pickle.dump(pipeline, f)


In [4]:
import joblib
filename_joblib = f'model_v{version}.joblib'
joblib.dump(pipeline, filename_joblib)


['model_v1.0.0.joblib']

In [5]:
with open(filename_pkl, 'rb') as f:
    model_pickle = pickle.load(f)
preds_pickle = model_pickle.predict(X_test)


In [6]:
with open(filename_pkl, 'rb') as f:
    model_pickle = pickle.load(f)
preds_pickle = model_pickle.predict(X_test)


In [7]:
model_joblib = joblib.load(filename_joblib)
preds_joblib = model_joblib.predict(X_test)


In [8]:
from sklearn.metrics import accuracy_score
print(accuracy_score(y_test, preds_pickle))
print(accuracy_score(y_test, preds_joblib))


0.9649122807017544
0.9649122807017544


In [9]:
import os
versions = ['1.0.0', '1.1.0', '2.0.0']
for v in versions:
    fname = f'model_v{v}.joblib'
    joblib.dump(pipeline, fname)
os.listdir('.')


['app.py',
 'model_v1.0.0.joblib',
 'model_v1.0.0.pkl',
 'model_v1.1.0.joblib',
 'model_v2.0.0.joblib',
 'Week_4_Day_3_Part2.ipynb']

In [10]:
import json
metadata = {
    'model_name': 'RandomForestPipeline',
    'versions': versions,
    'date_saved': {v: f'model_v{v}.joblib' for v in versions}
}
with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f)


In [11]:
with open('model_metadata.json', 'r') as f:
    meta = json.load(f)
print(meta)


{'model_name': 'RandomForestPipeline', 'versions': ['1.0.0', '1.1.0', '2.0.0'], 'date_saved': {'1.0.0': 'model_v1.0.0.joblib', '1.1.0': 'model_v1.1.0.joblib', '2.0.0': 'model_v2.0.0.joblib'}}


Excercise

### Overview  
This exercise guides you through training a scikit-learn pipeline, saving it with both Pickle and Joblib, managing model versions and metadata, and exposing the model via a simple Flask API  ([9. Model persistence — scikit-learn 1.6.1 documentation](https://scikit-learn.org/stable/model_persistence.html?utm_source=chatgpt.com)).  

### Tasks  
1. **Data Preparation:** Load the Breast Cancer Wisconsin dataset (`sklearn.datasets.load_breast_cancer`), split into train/test (80/20), and standardize features  ([Save and Load Machine Learning Models in Python with scikit-learn](https://machinelearningmastery.com/save-load-machine-learning-models-python-scikit-learn/?utm_source=chatgpt.com)).  
2. **Pipeline Construction:** Build an `sklearn.pipeline.Pipeline` with `StandardScaler` and `RandomForestClassifier(n_estimators=100, random_state=42)` and fit it on the training data  ([How to properly pickle sklearn pipeline when using custom ...](https://stackoverflow.com/questions/57888291/how-to-properly-pickle-sklearn-pipeline-when-using-custom-transformer?utm_source=chatgpt.com)).  
3. **Serialization:**  
   - Save the fitted pipeline using Pickle protocol 5 to `model_v1.pkl`.  
   - Save the same pipeline using Joblib to `model_v1.joblib`  ([Save Machine Learning Model Using Pickle and Joblib](https://www.analyticsvidhya.com/blog/2021/08/quick-hacks-to-save-machine-learning-model-using-pickle-and-joblib/?utm_source=chatgpt.com)).  
4. **Version Control:** Simulate versioning by retraining the pipeline with a different random seed, then saving as `model_v2.pkl`/`.joblib`; repeat for a third version. Create a `model_metadata.json` cataloging versions, file names, and timestamps  ([Machine Learning Model Serialization - Christopher Flynn](https://flynn.gg/blog/machine-learning-model-serialization/?utm_source=chatgpt.com)).  
5. **Deserialization & Validation:**  
   - Load each Pickle and Joblib file, run `.predict()` on the test set, and report accuracy.  
   - Measure and compare file sizes and load times between Pickle and Joblib  ([Pickle is over 10 times faster than joblib for save and load scikit ...](https://www.reddit.com/r/Python/comments/ypj13g/pickle_is_over_10_times_faster_than_joblib_for/?utm_source=chatgpt.com)).  
6. **Model Serving (Bonus):**  
   - Implement a minimal Flask app with two endpoints:  
     - `GET /models` returns available version metadata.  
     - `POST /predict` accepts JSON feature vectors and returns predicted classes using the latest model.  
   - Demonstrate sending requests via `requests` in Colab  ([Model Serialization using pickle and joblib - Kaggle](https://www.kaggle.com/code/tasnimniger/model-serialization-using-pickle-and-joblib?utm_source=chatgpt.com)).  

### Deliverables  
- A Colab notebook containing all code cells above.  
- The saved model files (`.pkl`, `.joblib`) and `model_metadata.json`.  
- A brief report (markdown cell) comparing Pickle vs Joblib in speed and size.  
- (Optional) The Flask app code cell demonstrating model serving.

In [12]:
# 1 Data Preparation

# Load dataset
data = load_breast_cancer()

# Features and target
X = data.data
y = data.target

# Split dataset (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Check the shapes
print("Training Features:", X_train.shape)
print("Testing Features :", X_test.shape)
print("Training Labels  :", y_train.shape)
print("Testing Labels   :", y_test.shape)

Training Features: (455, 30)
Testing Features : (114, 30)
Training Labels  : (455,)
Testing Labels   : (114,)


In [13]:
#2 Pipeline Construction

# Create the pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

# Train the pipeline
pipeline.fit(X_train, y_train)

print("Pipeline trained successfully!")

Pipeline trained successfully!


In [14]:
# 3 Serialization

# Save pipeline using Pickle (Protocol 5)
with open("model_v1.pkl", "wb") as file:
    pickle.dump(pipeline, file, protocol=5)

print("Pipeline saved as model_v1.pkl")

# Save pipeline using Joblib
joblib.dump(pipeline, "model_v1.joblib")

print("Pipeline saved as model_v1.joblib")

Pipeline saved as model_v1.pkl
Pipeline saved as model_v1.joblib


In [15]:
# Version 2 Pipeline
pipeline_v2 = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=100
    ))
])

# Train Version 2
pipeline_v2.fit(X_train, y_train)

# Save using Pickle
with open("model_v2.pkl", "wb") as file:
    pickle.dump(pipeline_v2, file, protocol=5)

# Save using Joblib
joblib.dump(pipeline_v2, "model_v2.joblib")

print("Version 2 saved successfully!")

Version 2 saved successfully!


In [16]:
# Version 3 Pipeline
pipeline_v3 = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=200
    ))
])

# Train Version 3
pipeline_v3.fit(X_train, y_train)

# Save using Pickle
with open("model_v3.pkl", "wb") as file:
    pickle.dump(pipeline_v3, file, protocol=5)

# Save using Joblib
joblib.dump(pipeline_v3, "model_v3.joblib")

print("Version 3 saved successfully!")

Version 3 saved successfully!


In [17]:
from datetime import datetime
metadata = {
    "models": [
        {
            "version": "v1",
            "pickle_file": "model_v1.pkl",
            "joblib_file": "model_v1.joblib",
            "timestamp": datetime.now().isoformat()
        },
        {
            "version": "v2",
            "pickle_file": "model_v2.pkl",
            "joblib_file": "model_v2.joblib",
            "timestamp": datetime.now().isoformat()
        },
        {
            "version": "v3",
            "pickle_file": "model_v3.pkl",
            "joblib_file": "model_v3.joblib",
            "timestamp": datetime.now().isoformat()
        }
    ]
}

with open("model_metadata.json", "w") as file:
    json.dump(metadata, file, indent=4)

print("Metadata file created successfully!")

Metadata file created successfully!


In [18]:
# 5 Deserialization & Validation
import time

# Files to evaluate
model_files = [
    "model_v1.pkl",
    "model_v1.joblib",
    "model_v2.pkl",
    "model_v2.joblib",
    "model_v3.pkl",
    "model_v3.joblib"
]

print("-" * 75)
print(f"{'Model':<18}{'Accuracy':<12}{'Size (KB)':<12}{'Load Time (ms)':<15}")
print("-" * 75)

for file_name in model_files:

    # Measure loading time
    start_time = time.perf_counter()

    if file_name.endswith(".pkl"):
        with open(file_name, "rb") as file:
            model = pickle.load(file)
    else:
        model = joblib.load(file_name)

    end_time = time.perf_counter()

    # Load time in milliseconds
    load_time = (end_time - start_time) * 1000

    # Make predictions
    predictions = model.predict(X_test)

    # Calculate accuracy
    accuracy = accuracy_score(y_test, predictions)

    # File size in KB
    file_size = os.path.getsize(file_name) / 1024

    # Display results
    print(f"{file_name:<18}{accuracy:.4f}      {file_size:.2f}      {load_time:.2f}")

---------------------------------------------------------------------------
Model             Accuracy    Size (KB)   Load Time (ms) 
---------------------------------------------------------------------------
model_v1.pkl      0.9561      307.32      640.27
model_v1.joblib   0.9561      317.85      321.40
model_v2.pkl      0.9474      304.98      45.52
model_v2.joblib   0.9474      315.50      242.48
model_v3.pkl      0.9474      307.48      40.84
model_v3.joblib   0.9474      318.00      90.39


TODO 6

In [20]:
# Test GET /models
import requests

response = requests.get("http://127.0.0.1:5000/models")

print(response.status_code)
print(response.json())

200
{'models': [{'joblib_file': 'model_v1.joblib', 'pickle_file': 'model_v1.pkl', 'timestamp': '2026-07-27T06:17:19.772859', 'version': 'v1'}, {'joblib_file': 'model_v2.joblib', 'pickle_file': 'model_v2.pkl', 'timestamp': '2026-07-27T06:17:19.772859', 'version': 'v2'}, {'joblib_file': 'model_v3.joblib', 'pickle_file': 'model_v3.pkl', 'timestamp': '2026-07-27T06:17:19.772859', 'version': 'v3'}]}


In [21]:
# Test POST /predict
import requests

sample = X_test[:1].tolist()

response = requests.post(
    "http://127.0.0.1:5000/predict",
    json={
        "features": sample
    }
)

print(response.status_code)
print(response.json())

200
{'predictions': [0]}
